# Chapter 11 — Regression Hedging and Principal Component Analysis

教材 pp. 136–145；PDF 页序为教材页码加 11。



下列未明确标为教材例题的数据均为合成验证数据，不声称复现教材中的原始市场样本。

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from numpy.testing import assert_allclose
from hedging_calculator import *

OLS 已知系数、对冲敞口抵消；pp.137–140。

In [2]:
x=np.arange(10.)
data=pd.DataFrame({'x':x,'y':.06+.842*x})
model=RegressionHedge(data,'y',['x'])
assert_allclose(model.coefficients,[.06,.842])
assert_allclose(model.predict(data),data.y)
assert_allclose(model.hedge_positions(100,pd.DataFrame({'dv01':[50]}),'dv01'),[-1.684])

PCA 保持总方差、完整分量能重建数据；pp.142–145。

In [3]:
rng=np.random.default_rng(7)
data=pd.DataFrame(rng.normal(size=(500,3))@np.array([[1,2,3],[0,1,2],[0,0,1]]),columns=['a','b','c'])
pca=PrincipalComponents(data,['a','b','c'])
scores=pca.transform(data,3)
assert_allclose(scores@pca.eigenvectors.T+pca.mean.to_numpy(),data,atol=1e-12)
assert_allclose(pca.eigenvalues.sum(),np.trace(data.cov()))
assert_allclose(pca.explained_variance().sum(),1)
print(pca.explained_variance())

0    0.970169
1    0.025109
2    0.004721
Name: explained_variance, dtype: float64


## 因子敞口对冲恒等式；p.145

In [4]:
target=pd.DataFrame([[10.,20.]])
hedges=pd.DataFrame([[2.,1.],[1.,3.]])
positions=pca.hedge_positions(target,hedges)
assert_allclose(hedges.to_numpy().T@positions+target.to_numpy().ravel(),0,atol=1e-12)